# X Full-Archive Text Mining for MBG and Food Tray  
## Google Colab notebook for skripsi

Notebook ini disusun untuk tujuan skripsi:

**REDESIGN FOOD TRAY PROGRAM MAKAN BERGIZI GRATIS (MBG) DENGAN VOICE OF CUSTOMER BERBASIS SOCIAL MEDIA ANALYSIS**

## Tujuan notebook
Notebook ini dibagi menjadi **2 corpus**:

### Corpus A — MBG Umum
Dipakai untuk menjawab:
- isu MBG yang paling sering muncul apa saja?
- posisi/rank isu **food tray / ompreng** dibanding:
  - menu
  - regulasi
  - kondisi dapur
  - distribusi
  - keracunan / keamanan pangan

### Corpus B — Food Tray MBG
Dipakai untuk:
- analisis sentimen khusus food tray
- topic modeling dengan **BERTopic**
- transformasi topic menjadi **draft need statement**

## Catatan penting
1. Kalau kamu mau data **Agustus 2025 sampai Oktober 2025**, kamu harus pakai **full archive endpoint**:
   `https://api.x.com/2/tweets/search/all`

2. Kalau akun X API kamu **tidak punya akses** ke endpoint itu, request akan gagal.
3. Token yang pernah kamu tampilkan di chat/screenshot sebaiknya **di-regenerate**.
4. Notebook ini dibuat untuk **bahasa Indonesia** dan fokus pada topik MBG.

In [ ]:
!pip -q install requests pandas openpyxl matplotlib tqdm Sastrawi bertopic sentence-transformers umap-learn hdbscan scikit-learn

## 1. Import package

In [ ]:
import os
import re
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from collections import Counter
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

## 2. Konfigurasi utama

In [ ]:
BEARER_TOKEN = "PASTE_BEARER_TOKEN_X_DI_SINI"

# Full archive untuk Agustus 2025 - Oktober 2025
SEARCH_URL = "https://api.x.com/2/tweets/search/all"
# SEARCH_URL = "https://api.x.com/2/tweets/search/recent"  # hanya untuk test 7 hari terakhir

START_TIME = "2025-08-01T00:00:00Z"
END_TIME   = "2025-10-31T23:59:59Z"

OUTPUT_DIR = "/content/output_mbg_x_foodtray"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "mbg_x_foodtray_analysis.xlsx")

MAX_RESULTS_PER_PAGE = 100
MAX_PAGES_PER_QUERY = 30
SLEEP_BETWEEN_REQUESTS = 1.2

TWEET_FIELDS = "id,text,created_at,lang,public_metrics,author_id,conversation_id,referenced_tweets"
USER_FIELDS = "id,username,name,verified,public_metrics"
EXPANSIONS = "author_id"

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not BEARER_TOKEN.strip():
    raise ValueError("Bearer token X belum diisi.")

print("Endpoint  :", SEARCH_URL)
print("Start time:", START_TIME)
print("End time  :", END_TIME)
print("Token OK? :", bool(BEARER_TOKEN.strip()))
print("Token head:", BEARER_TOKEN[:12] + "...")

## 3. Query design

In [ ]:
QUERY_PACKS_GENERAL = {
    "mbg_umum_phrase": '"makan bergizi gratis" lang:id -is:retweet',
    "mbg_umum_short": 'MBG lang:id -is:retweet',
    "mbg_umum_context": '(MBG OR "makan bergizi gratis" OR SPPG OR "program makan bergizi gratis") lang:id -is:retweet'
}

OBJECT_BASE = (
    '(("food tray MBG" OR "tray MBG" OR "wadah MBG" OR "tempat makan MBG" OR "ompreng MBG") '
    'OR (ompreng (MBG OR "makan bergizi gratis")) '
    'OR ("food tray" (MBG OR "makan bergizi gratis")) '
    'OR (wadah (MBG OR "makan bergizi gratis")) '
    'OR (tray (MBG OR "makan bergizi gratis")))'
)

QUERY_PACKS_TRAY = {
    "tray_umum_objek": f'{OBJECT_BASE} lang:id -is:retweet',
    "tray_suhu_basi": f'{OBJECT_BASE} (dingin OR panas OR hangat OR suhu OR basi OR "cepat basi") lang:id -is:retweet',
    "tray_tumpah_tutup": f'{OBJECT_BASE} (tumpah OR bocor OR tutup OR rapat OR kuah) lang:id -is:retweet',
    "tray_material": f'{OBJECT_BASE} (stainless OR "ss 304" OR "food grade" OR karat OR "anti karat" OR bahan) lang:id -is:retweet',
    "tray_distribusi": f'{OBJECT_BASE} (distribusi OR angkut OR ikat OR diikat OR rafia OR rapia OR tali OR tumpuk OR handling) lang:id -is:retweet',
    "tray_higienitas": f'{OBJECT_BASE} (higienis OR sanitasi OR bersih OR kotor OR kontaminasi OR cuci OR kering) lang:id -is:retweet',
}

print("Jumlah query MBG umum :", len(QUERY_PACKS_GENERAL))
print("Jumlah query tray MBG :", len(QUERY_PACKS_TRAY))

## 4. Function untuk fetch data dari X API

In [ ]:
HEADERS = {"Authorization": f"Bearer {BEARER_TOKEN}"}

def fetch_x_search(query, search_url, start_time, end_time, max_results=100, max_pages=10, sleep_sec=1.2):
    rows = []
    next_token = None
    page = 0

    while page < max_pages:
        params = {
            "query": query,
            "max_results": max_results,
            "tweet.fields": TWEET_FIELDS,
            "user.fields": USER_FIELDS,
            "expansions": EXPANSIONS,
        }

        if start_time:
            params["start_time"] = start_time
        if end_time:
            params["end_time"] = end_time
        if next_token:
            params["next_token"] = next_token

        resp = requests.get(search_url, headers=HEADERS, params=params, timeout=60)

        if resp.status_code != 200:
            print("ERROR STATUS:", resp.status_code)
            print("ERROR BODY  :", resp.text[:1500])
            break
        else:
            print(f"STATUS OK: {resp.status_code} | page {page+1}")

        payload = resp.json()
        tweets = payload.get("data", [])
        includes = payload.get("includes", {})
        users = includes.get("users", [])
        user_map = {u["id"]: u for u in users}

        if not tweets:
            print("No tweets on this page.")
            break

        for tw in tweets:
            metrics = tw.get("public_metrics", {})
            author = user_map.get(tw.get("author_id"), {})
            rows.append({
                "tweet_id": tw.get("id"),
                "conversation_id": tw.get("conversation_id"),
                "created_at": tw.get("created_at"),
                "author_id": tw.get("author_id"),
                "author_name": author.get("name"),
                "author_username": author.get("username"),
                "author_verified": author.get("verified"),
                "lang": tw.get("lang"),
                "text_raw": tw.get("text"),
                "like_count": metrics.get("like_count", 0),
                "reply_count": metrics.get("reply_count", 0),
                "repost_count": metrics.get("retweet_count", 0),
                "quote_count": metrics.get("quote_count", 0),
                "bookmark_count": metrics.get("bookmark_count", 0),
                "query_used": query,
                "tweet_url": f'https://x.com/{author.get("username","unknown")}/status/{tw.get("id")}'
            })

        next_token = payload.get("meta", {}).get("next_token")
        page += 1

        if not next_token:
            break

        time.sleep(sleep_sec)

    return pd.DataFrame(rows)

def collect_query_pack(query_pack, label_prefix):
    dfs = []
    stats = []

    for label, q in query_pack.items():
        print("\n" + "="*80)
        print("LABEL:", label)
        print("QUERY:", q)

        df_q = fetch_x_search(
            query=q,
            search_url=SEARCH_URL,
            start_time=START_TIME,
            end_time=END_TIME,
            max_results=MAX_RESULTS_PER_PAGE,
            max_pages=MAX_PAGES_PER_QUERY,
            sleep_sec=SLEEP_BETWEEN_REQUESTS
        )

        n = len(df_q)
        print("TOTAL ROWS:", n)

        stats.append({
            "query_group": label_prefix,
            "query_label": label,
            "rows_retrieved": n
        })

        if not df_q.empty:
            df_q["query_group"] = label_prefix
            df_q["query_label"] = label
            dfs.append(df_q)

    if dfs:
        out = pd.concat(dfs, ignore_index=True)
        out = out.drop_duplicates(subset=["tweet_id"]).reset_index(drop=True)
    else:
        out = pd.DataFrame()

    return out, pd.DataFrame(stats)

## 5. Test akses endpoint

In [ ]:
test_query = '"makan bergizi gratis" lang:id -is:retweet'
test_df = fetch_x_search(
    query=test_query,
    search_url=SEARCH_URL,
    start_time=START_TIME,
    end_time=END_TIME,
    max_results=10,
    max_pages=1,
    sleep_sec=0.5
)

print("Test rows:", len(test_df))
display(test_df.head())

## 6. Ambil Corpus A — MBG umum

In [ ]:
general_df, general_stats_df = collect_query_pack(QUERY_PACKS_GENERAL, label_prefix="general")
print("GENERAL ROWS:", len(general_df))
display(general_stats_df)
display(general_df.head())

## 7. Ambil Corpus B — Food Tray MBG

In [ ]:
tray_df, tray_stats_df = collect_query_pack(QUERY_PACKS_TRAY, label_prefix="tray")
print("TRAY ROWS:", len(tray_df))
display(tray_stats_df)
display(tray_df.head())

## 8. Simpan data mentah

In [ ]:
general_df.to_csv(os.path.join(OUTPUT_DIR, "01_general_raw.csv"), index=False)
tray_df.to_csv(os.path.join(OUTPUT_DIR, "02_tray_raw.csv"), index=False)
general_stats_df.to_csv(os.path.join(OUTPUT_DIR, "00_general_query_stats.csv"), index=False)
tray_stats_df.to_csv(os.path.join(OUTPUT_DIR, "00_tray_query_stats.csv"), index=False)
print("Raw files saved.")

## 9. Preprocessing

In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

CUSTOM_STOPWORDS = {
    "yang","dan","di","ke","dari","untuk","atau","pada","dengan","karena","jadi","itu","ini",
    "aja","juga","udah","sudah","nya","ya","kok","nih","sih","mah","dong",
    "rt","via","yg","utk","dr","krn","tp","tapi","dll","deh",
    "mbg","makan","bergizi","gratis","program","sppg"
}

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"&amp;", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return [w for w in text.split() if len(w) > 2 and w not in CUSTOM_STOPWORDS]

def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]

def prep_dataframe(df):
    out = df.copy()
    out["text_clean"] = out["text_raw"].fillna("").apply(clean_text)
    out["tokens"] = out["text_clean"].apply(tokenize)
    out["tokens_stem"] = out["tokens"].apply(stem_tokens)
    out["text_joined"] = out["tokens_stem"].apply(lambda x: " ".join(x))
    out = out[out["text_joined"].str.len() > 0].copy()
    out = out.drop_duplicates(subset=["text_joined"]).reset_index(drop=True)
    return out

general_pp = prep_dataframe(general_df)
tray_pp = prep_dataframe(tray_df)

print("general after prep:", len(general_pp))
print("tray after prep   :", len(tray_pp))

display(general_pp[["text_raw","text_joined"]].head())
display(tray_pp[["text_raw","text_joined"]].head())

## 10. Ranking isu pada Corpus A — MBG umum

In [ ]:
ISSUE_DICT = {
    "food_tray_ompreng": [
        "food tray","tray","ompreng","wadah","tempat makan","alat makan","tutup","bocor","tumpah","rafia","stainless"
    ],
    "menu_gizi": [
        "menu","rasa","lauk","sayur","nasi","buah","susu","gizi","protein","porsi","variasi menu"
    ],
    "regulasi_kebijakan": [
        "regulasi","aturan","kebijakan","perpres","juknis","sni","standar","pengawasan","bpom","bsn"
    ],
    "kondisi_dapur_spgg": [
        "dapur","spgg","kitchen","alat masak","kebersihan dapur","sanitasi dapur","fasilitas","pegawai dapur","juru masak"
    ],
    "distribusi_logistik": [
        "distribusi","logistik","angkut","kirim","telat","terlambat","jarak","handling","pengiriman","transportasi"
    ]
}

def count_issue_hits(text, issue_dict):
    hits = []
    for issue, kws in issue_dict.items():
        for kw in kws:
            if kw in text:
                hits.append(issue)
                break
    return hits

general_pp["issue_hits"] = general_pp["text_clean"].apply(lambda x: count_issue_hits(x, ISSUE_DICT))
general_issue = general_pp[general_pp["issue_hits"].apply(len) > 0].copy().reset_index(drop=True)

issue_counter = Counter()
for lst in general_issue["issue_hits"]:
    issue_counter.update(lst)

issue_rank_df = pd.DataFrame(
    [{"issue": k, "tweet_count": v} for k, v in issue_counter.items()]
).sort_values("tweet_count", ascending=False).reset_index(drop=True)

if len(general_issue) > 0:
    issue_rank_df["share_of_issue_coded_tweets_pct"] = (
        issue_rank_df["tweet_count"] / len(general_issue) * 100
    ).round(2)
else:
    issue_rank_df["share_of_issue_coded_tweets_pct"] = 0.0

issue_rank_df["rank"] = range(1, len(issue_rank_df) + 1)
issue_rank_df = issue_rank_df[["rank", "issue", "tweet_count", "share_of_issue_coded_tweets_pct"]]

display(issue_rank_df)

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(issue_rank_df["issue"], issue_rank_df["tweet_count"])
plt.xticks(rotation=35, ha="right")
plt.title("Ranking Isu MBG pada Corpus Umum")
plt.ylabel("Jumlah Tweet")
plt.show()

## 11. Kata dominan pada Corpus A — MBG umum

In [ ]:
general_tokens = []
for toks in general_pp["tokens_stem"]:
    general_tokens.extend(toks)

general_top_words = pd.DataFrame(
    Counter(general_tokens).most_common(100),
    columns=["word", "count"]
)

display(general_top_words.head(30))

## 12. Sentimen sederhana pada Corpus B — Food Tray MBG

In [ ]:
POSITIVE_WORDS = ["bagus","baik","aman","rapi","kuat","bersih","higienis","sesuai","mantap","oke","layak"]
NEGATIVE_WORDS = [
    "dingin","basi","tumpah","bocor","kotor","kurang","buruk","karat","rusak","berat","susah",
    "ribet","takut","bahaya","kontaminasi","jelek","tipis","kendala","masalah","tidak rapat",
    "gak rapat","ga rapat","nggak rapat","ga rapet","ga bersih","gak bersih"
]

def sentiment_score(text):
    pos = sum(1 for w in POSITIVE_WORDS if w in text)
    neg = sum(1 for w in NEGATIVE_WORDS if w in text)
    return pos - neg

def sentiment_label(score):
    if score > 0:
        return "positif"
    elif score < 0:
        return "negatif"
    return "netral"

tray_pp["sentiment_score"] = tray_pp["text_clean"].apply(sentiment_score)
tray_pp["sentiment_label"] = tray_pp["sentiment_score"].apply(sentiment_label)

sent_summary = tray_pp["sentiment_label"].value_counts().reset_index()
sent_summary.columns = ["sentiment", "count"]

display(sent_summary)

plt.figure(figsize=(6,4))
plt.bar(sent_summary["sentiment"], sent_summary["count"])
plt.title("Sentimen pada Corpus Food Tray MBG")
plt.ylabel("Jumlah Tweet")
plt.show()

## Analisis n-gram dan Co-occurrence

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from itertools import combinations
import numpy as np

In [ ]:
def get_top_ngrams(text_series, ngram_range=(2,2), top_n=30, min_df=2):
    texts = text_series.fillna("").astype(str).tolist()

    vectorizer = CountVectorizer(
        ngram_range=ngram_range,
        min_df=min_df
    )

    X = vectorizer.fit_transform(texts)
    terms = vectorizer.get_feature_names_out()
    counts = np.asarray(X.sum(axis=0)).ravel()

    ngram_df = pd.DataFrame({
        "ngram": terms,
        "count": counts
    }).sort_values("count", ascending=False).reset_index(drop=True)

    return ngram_df.head(top_n), ngram_df

# =========================
# GENERAL CORPUS
# =========================
general_top_bigrams, general_bigrams_all = get_top_ngrams(
    general_pp["text_joined"],
    ngram_range=(2,2),
    top_n=30,
    min_df=2
)

general_top_trigrams, general_trigrams_all = get_top_ngrams(
    general_pp["text_joined"],
    ngram_range=(3,3),
    top_n=30,
    min_df=2
)

# =========================
# TRAY CORPUS
# =========================
tray_top_bigrams, tray_bigrams_all = get_top_ngrams(
    tray_pp["text_joined"],
    ngram_range=(2,2),
    top_n=30,
    min_df=2
)

tray_top_trigrams, tray_trigrams_all = get_top_ngrams(
    tray_pp["text_joined"],
    ngram_range=(3,3),
    top_n=30,
    min_df=2
)

print("TOP BIGRAM MBG UMUM")
display(general_top_bigrams)

print("TOP TRIGRAM MBG UMUM")
display(general_top_trigrams)

print("TOP BIGRAM FOOD TRAY")
display(tray_top_bigrams)

print("TOP TRIGRAM FOOD TRAY")
display(tray_top_trigrams)

In [ ]:
def plot_top_ngrams(df_ngram, title, top_n=15):
    plot_df = df_ngram.head(top_n).copy()
    plot_df = plot_df.iloc[::-1]

    plt.figure(figsize=(10,6))
    plt.barh(plot_df["ngram"], plot_df["count"])
    plt.title(title)
    plt.xlabel("Frekuensi")
    plt.ylabel("N-gram")
    plt.show()

plot_top_ngrams(general_top_bigrams, "Top Bigram pada Corpus MBG Umum", top_n=15)
plot_top_ngrams(tray_top_bigrams, "Top Bigram pada Corpus Food Tray MBG", top_n=15)
plot_top_ngrams(tray_top_trigrams, "Top Trigram pada Corpus Food Tray MBG", top_n=15)

In [ ]:
def build_cooccurrence_from_docs(text_series, min_df=3, max_features=20):
    vectorizer = CountVectorizer(
        ngram_range=(1,1),
        min_df=min_df,
        max_features=max_features,
        binary=True
    )

    X = vectorizer.fit_transform(text_series.fillna("").astype(str))
    terms = vectorizer.get_feature_names_out()

    co_matrix = (X.T @ X).toarray()
    np.fill_diagonal(co_matrix, 0)

    co_df = pd.DataFrame(co_matrix, index=terms, columns=terms)
    return co_df

tray_co_df = build_cooccurrence_from_docs(
    tray_pp["text_joined"],
    min_df=3,
    max_features=20
)

print("Co-occurrence matrix untuk corpus food tray")
display(tray_co_df)

In [ ]:
def extract_top_cooccurrence_pairs(co_df, top_n=30):
    rows = []
    terms = list(co_df.index)

    for i in range(len(terms)):
        for j in range(i + 1, len(terms)):
            count = co_df.iloc[i, j]
            if count > 0:
                rows.append({
                    "term_1": terms[i],
                    "term_2": terms[j],
                    "cooccurrence_count": int(count)
                })

    pair_df = pd.DataFrame(rows).sort_values(
        "cooccurrence_count",
        ascending=False
    ).reset_index(drop=True)

    return pair_df.head(top_n), pair_df

tray_top_pairs, tray_pairs_all = extract_top_cooccurrence_pairs(tray_co_df, top_n=30)

print("Top co-occurrence pairs pada corpus food tray")
display(tray_top_pairs)

In [ ]:
def plot_cooccurrence_heatmap(co_df, title="Co-occurrence Heatmap"):
    fig, ax = plt.subplots(figsize=(10,8))
    cax = ax.imshow(co_df.values, aspect="auto")

    ax.set_xticks(range(len(co_df.columns)))
    ax.set_yticks(range(len(co_df.index)))
    ax.set_xticklabels(co_df.columns, rotation=90)
    ax.set_yticklabels(co_df.index)

    ax.set_title(title)
    fig.colorbar(cax)
    plt.tight_layout()
    plt.show()

plot_cooccurrence_heatmap(tray_co_df, title="Co-occurrence Heatmap pada Corpus Food Tray MBG")

In [ ]:
DESIGN_TERMS = [
    "tray", "ompreng", "wadah", "tutup", "bocor", "tumpah",
    "kuah", "dingin", "basi", "suhu", "stainless", "karat",
    "bahan", "distribusi", "rafia", "ikat", "higienis",
    "sanitasi", "kontaminasi", "bersih"
]

def build_keyword_cooccurrence(df_in, terms):
    rows = []

    for text in df_in["text_clean"].fillna("").astype(str):
        present_terms = [t for t in terms if t in text]
        present_terms = sorted(list(set(present_terms)))

        for a, b in combinations(present_terms, 2):
            rows.append((a, b))

    pair_counter = Counter(rows)

    pair_df = pd.DataFrame([
        {"term_1": k[0], "term_2": k[1], "cooccurrence_count": v}
        for k, v in pair_counter.items()
    ]).sort_values("cooccurrence_count", ascending=False).reset_index(drop=True)

    # matrix
    term_index = {t: i for i, t in enumerate(terms)}
    matrix = np.zeros((len(terms), len(terms)), dtype=int)

    for _, row in pair_df.iterrows():
        i = term_index[row["term_1"]]
        j = term_index[row["term_2"]]
        matrix[i, j] = row["cooccurrence_count"]
        matrix[j, i] = row["cooccurrence_count"]

    matrix_df = pd.DataFrame(matrix, index=terms, columns=terms)
    return pair_df, matrix_df

tray_design_pairs_df, tray_design_matrix_df = build_keyword_cooccurrence(tray_pp, DESIGN_TERMS)

print("Top co-occurrence pairs berbasis keyword desain")
display(tray_design_pairs_df.head(30))

plot_cooccurrence_heatmap(
    tray_design_matrix_df,
    title="Co-occurrence Heatmap Berbasis Keyword Desain Food Tray MBG"
)

## 13. BERTopic untuk Corpus B — Food Tray MBG

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

seed_topic_list = [
    ["food tray", "tray", "ompreng", "wadah"],
    ["dingin", "hangat", "suhu", "basi", "cepat basi"],
    ["tumpah", "bocor", "tutup", "rapat", "kuah"],
    ["stainless", "ss 304", "food grade", "karat", "bahan"],
    ["distribusi", "angkut", "rafia","tali","rapia","ikat", "tumpuk", "handling"],
    ["higienis", "sanitasi", "bersih", "kotor", "kontaminasi", "cuci"]
]

docs_tray = tray_pp["text_clean"].fillna("").tolist()

if len(docs_tray) < 30:
    raise ValueError("Jumlah dokumen untuk BERTopic terlalu sedikit. Tambah data atau longgarkan query.")

min_topic_size = 8 if len(docs_tray) < 300 else 10

vectorizer_model = CountVectorizer(
    ngram_range=(1,2),
    min_df=2
)

topic_model = BERTopic(
    language="multilingual",
    seed_topic_list=seed_topic_list,
    min_topic_size=min_topic_size,
    calculate_probabilities=True,
    representation_model=KeyBERTInspired(),
    vectorizer_model=vectorizer_model
)

topics, probs = topic_model.fit_transform(docs_tray)

tray_pp["topic"] = topics
topic_info = topic_model.get_topic_info()

display(topic_info.head(20))

## 14. Visualisasi topic

In [ ]:
topic_info_non_outlier = topic_info[topic_info["Topic"] != -1].copy()

plt.figure(figsize=(10,5))
plt.bar(topic_info_non_outlier["Name"], topic_info_non_outlier["Count"])
plt.xticks(rotation=45, ha="right")
plt.title("BERTopic Topics for Food Tray MBG")
plt.ylabel("Jumlah Tweet")
plt.show()

## 15. Representative tweets per topic

In [ ]:
def representative_docs_per_topic(df_in, topic_col="topic", top_n=5):
    rows = []
    for topic_id in sorted(df_in[topic_col].dropna().unique()):
        if topic_id == -1:
            continue
        temp = df_in[df_in[topic_col] == topic_id].copy()
        if temp.empty:
            continue

        temp["rep_score"] = temp["like_count"] + 2*temp["reply_count"] + 2*temp["repost_count"]
        top_rows = temp.sort_values("rep_score", ascending=False).head(top_n)

        for _, r in top_rows.iterrows():
            rows.append({
                "topic": topic_id,
                "text_raw": r["text_raw"],
                "tweet_url": r["tweet_url"],
                "rep_score": r["rep_score"]
            })
    return pd.DataFrame(rows)

topic_rep_df = representative_docs_per_topic(tray_pp, topic_col="topic", top_n=5)
display(topic_rep_df.head(20))

## 16. Draft need statement dari topic

In [ ]:
topic_keywords_map = {}
for topic_id in topic_info_non_outlier["Topic"].tolist():
    kws = topic_model.get_topic(topic_id)
    topic_keywords_map[topic_id] = [w for w, _ in kws[:10]] if kws else []

def draft_need_statement(keywords):
    text = " ".join(keywords)

    if any(k in text for k in ["dingin", "hangat", "suhu", "basi"]):
        return "Food tray harus mampu memperlambat penurunan suhu makanan selama distribusi."
    if any(k in text for k in ["tumpah", "bocor", "tutup", "rapat", "kuah"]):
        return "Food tray harus mampu meminimalkan risiko kebocoran dan tumpahan selama pengangkutan dan distribusi."
    if any(k in text for k in ["stainless", "food", "karat", "bahan"]):
        return "Food tray harus menggunakan material yang aman pangan, tahan korosi, dan sesuai untuk distribusi MBG."
    if any(k in text for k in ["higien", "sanitasi", "bersih", "kotor", "kontaminasi", "cuci"]):
        return "Food tray harus mudah dibersihkan dan mendukung higienitas selama penanganan dan distribusi."
    if any(k in text for k in ["distribusi", "angkut", "rafia","rapia", "ikat", "handling", "tumpuk"]):
        return "Food tray harus stabil saat dibawa, mudah ditumpuk, dan mendukung handling distribusi yang aman."
    return "Food tray harus dirancang agar lebih sesuai dengan kebutuhan distribusi MBG di lapangan."

need_rows = []
for topic_id, kws in topic_keywords_map.items():
    count_topic = int((tray_pp["topic"] == topic_id).sum())
    need_rows.append({
        "topic": topic_id,
        "keywords": ", ".join(kws),
        "tweet_count": count_topic,
        "draft_need_statement": draft_need_statement(kws),
        "manual_final_need_statement": "",
        "manual_notes": ""
    })

need_statement_df = pd.DataFrame(need_rows).sort_values("tweet_count", ascending=False).reset_index(drop=True)
display(need_statement_df)

## 17. Manual review template

In [ ]:
manual_review_cols = [
    "tweet_id","created_at","author_username","text_raw","text_clean",
    "sentiment_label","topic","tweet_url"
]

manual_review_df = tray_pp[manual_review_cols].copy()
manual_review_df["manual_keep"] = ""
manual_review_df["manual_topic_label"] = ""
manual_review_df["manual_issue_category"] = ""
manual_review_df["manual_design_implication"] = ""
manual_review_df["manual_notes"] = ""

display(manual_review_df.head(20))

## 18. Simpan semua output

In [ ]:
general_pp.to_csv(os.path.join(OUTPUT_DIR, "03_general_preprocessed.csv"), index=False)
tray_pp.to_csv(os.path.join(OUTPUT_DIR, "04_tray_preprocessed.csv"), index=False)
issue_rank_df.to_csv(os.path.join(OUTPUT_DIR, "05_issue_ranking_general.csv"), index=False)
general_top_words.to_csv(os.path.join(OUTPUT_DIR, "06_general_top_words.csv"), index=False)
sent_summary.to_csv(os.path.join(OUTPUT_DIR, "07_tray_sentiment_summary.csv"), index=False)
topic_info.to_csv(os.path.join(OUTPUT_DIR, "08_tray_topic_info.csv"), index=False)
topic_rep_df.to_csv(os.path.join(OUTPUT_DIR, "09_tray_topic_representative_docs.csv"), index=False)
need_statement_df.to_csv(os.path.join(OUTPUT_DIR, "10_tray_need_statements_draft.csv"), index=False)
manual_review_df.to_csv(os.path.join(OUTPUT_DIR, "11_tray_manual_review.csv"), index=False)

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    general_stats_df.to_excel(writer, sheet_name="query_stats_general", index=False)
    tray_stats_df.to_excel(writer, sheet_name="query_stats_tray", index=False)
    general_df.to_excel(writer, sheet_name="general_raw", index=False)
    tray_df.to_excel(writer, sheet_name="tray_raw", index=False)
    general_pp.to_excel(writer, sheet_name="general_preprocessed", index=False)
    tray_pp.to_excel(writer, sheet_name="tray_preprocessed", index=False)
    issue_rank_df.to_excel(writer, sheet_name="issue_ranking_general", index=False)
    general_top_words.to_excel(writer, sheet_name="general_top_words", index=False)
    sent_summary.to_excel(writer, sheet_name="tray_sentiment_summary", index=False)
    topic_info.to_excel(writer, sheet_name="tray_topic_info", index=False)
    topic_rep_df.to_excel(writer, sheet_name="tray_topic_rep_docs", index=False)
    need_statement_df.to_excel(writer, sheet_name="draft_need_statement", index=False)
    manual_review_df.to_excel(writer, sheet_name="manual_review_tray", index=False)

print("Saved to:", OUTPUT_FILE)
print("Output dir:", OUTPUT_DIR)
print("Files:", sorted(os.listdir(OUTPUT_DIR)))

## 19. Cara membaca hasil untuk skripsi

### Untuk menjawab rank issue
Lihat sheet:
- `issue_ranking_general`

### Untuk menjawab kebutuhan desain
Lihat sheet:
- `tray_topic_info`
- `tray_topic_rep_docs`
- `draft_need_statement`
- `manual_review_tray`

### Alur ilmiah untuk sidang
1. Corpus MBG umum dipakai untuk memetakan posisi isu food tray terhadap isu MBG lain
2. Corpus food tray dipakai untuk mendalami voice of customer yang spesifik pada wadah/tray
3. BERTopic dipakai untuk mengelompokkan tema keluhan
4. Topik-topik tersebut diverifikasi secara manual
5. Hasilnya diterjemahkan menjadi need statement untuk redesign food tray MBG